# YOLOv8n-seg Training — CountingChallenge

This notebook trains a YOLOv8n-seg model on custom annotated screw/nut/bolt dataset.

**Run on:** Google Colab (T4 GPU)  
**Final model:** `best.pt` — 96.3% counting accuracy on 29 test images

## Dataset
- 25 images manually annotated in LabelMe (polygon masks)
- Classes: `nut`, `screw`
- Augmentation: flip_lr, flip_ud, rot90, brightness → **125 train + 25 val images**
- Original image size: ~2212×1659px, resized to 1280px for training

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics

In [ ]:
# Step 2: Upload and extract dataset
# Upload yolov8s_300_dataset.zip to Colab first, then:
import zipfile, shutil
from pathlib import Path

shutil.rmtree('/content/dataset', ignore_errors=True)
with zipfile.ZipFile('/content/yolov8s_300_dataset.zip') as z:
    z.extractall('/content/dataset')

# Verify
train_imgs = len(list(Path('/content/dataset/images/train').glob('*.jpg')))
val_imgs   = len(list(Path('/content/dataset/images/val').glob('*.jpg')))
print(f'Train: {train_imgs} images, Val: {val_imgs} images')

In [ ]:
# Step 3: Train YOLOv8n-seg
import sys, unittest.mock as mock
sys.modules['mlflow'] = mock.MagicMock()

from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')  # start from pretrained nano seg

model.train(
    data='/content/dataset/data.yaml',
    epochs=10,
    imgsz=1280,          # high resolution for dense small objects
    batch=2,             # small batch due to T4 VRAM limit
    device='0',
    project='/content/runs',
    name='v4_corrected_seg',
    patience=10,
    augment=False,       # manual augmentation already applied
    mosaic=0.0,
    degrees=0, fliplr=0.0, flipud=0.0, scale=0.0,
    workers=2,
    mask_ratio=16,       # reduce mask memory for dense images
    overlap_mask=False,
    max_det=500,
)
print('Training complete!')

In [ ]:
# Step 4: Evaluate best model
from ultralytics import YOLO

best_model = YOLO('/content/runs/v4_corrected_seg/weights/best.pt')
metrics = best_model.val(data='/content/dataset/data.yaml', imgsz=1280)
print(f'Box mAP50   : {metrics.box.map50:.3f}')
print(f'Mask mAP50  : {metrics.seg.map50:.3f}')

In [ ]:
# Step 5: Download trained weights
from google.colab import files
files.download('/content/runs/v4_corrected_seg/weights/best.pt')
print('Downloaded best.pt — place it in AI/ folder for inference')

## Training Configuration Summary

| Parameter | Value | Reason |
|-----------|-------|--------|
| Base model | yolov8n-seg | Lightweight, fast |
| imgsz | 1280 | Dense small objects need high res |
| batch | 2 | T4 VRAM limit (14GB) |
| epochs | 10 | Sufficient with augmented data |
| mask_ratio | 16 | Reduces memory for 300+ instances |
| overlap_mask | False | Prevents OOM on dense nut images |
| augment | False | Manual augmentations already applied |

## Inference Configuration

| Parameter | Value |
|-----------|-------|
| conf | 0.4 |
| iou | 0.3 |
| imgsz | 1280 |
| max_det | 500 |

**Final accuracy: 96.3% average across 29 test images**